# Perform single-cell level quality control

In [1]:
import os
import pathlib
import sys

import pandas as pd
from cosmicqc import find_outliers
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    image_based_profiles_subparent_name = "image_based_profiles"

## Load in each single-cell level profile per patient and process

1. Load in the single-cell data (add `patient_id` column).
2. Load in respective organoid qc data (only metadata and cqc columns) to already flag cells that come from a flagged organoid.
   - Also add a flag for if single-cells do not have an organoid segmentation (`parent_organoid` == -1).
   - Also add flag for if the `object_id` for a single-cell is NaN.
3. Concat single-cell data together.

In [3]:
sc_file = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "3.annotated_profiles/sc_anno.parquet"
)
organoid_file = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles/organoid_flagged_outliers.parquet"
)

output_dir = pathlib.Path(
    profile_base_dir
    / "data"
    / f"{patient}"
    / f"{image_based_profiles_subparent_name}"
    / "4.qc_profiles"
)
output_dir.mkdir(parents=True, exist_ok=True)

orig_sc_profiles_df = pd.read_parquet(sc_file)
organoid_qc_profiles_df = pd.read_parquet(organoid_file)
# Print the shape and head of the combined organoid profiles DataFrame
print(orig_sc_profiles_df.shape)
orig_sc_profiles_df

(190, 2648)


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,Metadata_Location_Cell_CenterY,Metadata_Location_Cell_CenterZ,...,Nuclei_Mito_Texture_Variance-3-09-256,Nuclei_Mito_Texture_Variance-3-10-256,Nuclei_Mito_Texture_Variance-3-11-256,Nuclei_Mito_Texture_Variance-3-12-256,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_EquivalentDiameter,Nuclei_NoChannel_AreaSizeShape_EulerNumber,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei_NoChannel_AreaSizeShape_Volume
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1159.400905,258.335972,2.993921,...,0.0,0.0,-18.892036,-19.361521,80465.0,41.115579,1,0.452284,227.191787,36393.0
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,488.147784,430.622409,2.875093,...,0.0,0.0,-18.084681,-19.538399,85330.0,38.731811,1,0.356533,218.941201,30423.0
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,410.433901,1068.101440,1.147251,...,0.0,0.0,-19.307073,-19.974498,6688.0,20.848710,1,0.709480,24.215672,4745.0
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1224.321794,394.006757,3.063999,...,0.0,0.0,-18.892036,-19.635696,35500.0,34.298864,1,0.595127,111.924414,21127.0
4,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,420.393892,1091.808172,2.935847,...,0.0,0.0,-18.307073,-19.447251,64326.0,31.540664,0,0.255402,205.985812,16429.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
185,NF0014_T1,Control,Control,Control,DMSO 1%,G9,G9-1,707.562771,964.971727,34.018463,...,0.0,0.0,0.000000,0.000000,226920.0,65.881033,1,0.659792,594.694255,149720.0
186,NF0014_T1,Control,Control,Control,DMSO 1%,G9,G9-1,824.912118,835.517205,32.579058,...,0.0,0.0,0.000000,0.000000,100110.0,45.267535,1,0.485156,330.232195,48569.0
187,NF0014_T1,Control,Control,Control,DMSO 1%,G9,G9-1,1017.698350,926.913999,40.820542,...,0.0,0.0,0.000000,0.000000,118944.0,50.912004,1,0.580920,307.055593,69097.0
188,NF0014_T1,Control,Control,Control,DMSO 1%,G9,G9-1,931.722692,1027.569053,37.382332,...,0.0,0.0,0.000000,0.000000,142695.0,58.196451,1,0.723235,345.851959,103202.0


In [4]:
sc_profiles_df = orig_sc_profiles_df.copy()
sc_profiles_df["Metadata_cqc_nan_detected"] = (
    sc_profiles_df[
        [
            "Metadata_Object_ObjectID",
            "Metadata_Object_ParentOrganoid",
            "Cell_NoChannel_AreaSizeShape_Volume",
        ]
    ]
    .isna()
    .any(axis=1)
)
# Print the number of organoids flagged
flagged_count = sc_profiles_df["Metadata_cqc_nan_detected"].sum()
print(f"Number of organoids flagged: {flagged_count}")

sc_profiles_df.head()

Number of organoids flagged: 0


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,Metadata_Location_Cell_CenterY,Metadata_Location_Cell_CenterZ,...,Nuclei_Mito_Texture_Variance-3-10-256,Nuclei_Mito_Texture_Variance-3-11-256,Nuclei_Mito_Texture_Variance-3-12-256,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_EquivalentDiameter,Nuclei_NoChannel_AreaSizeShape_EulerNumber,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei_NoChannel_AreaSizeShape_Volume,Metadata_cqc_nan_detected
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1159.400905,258.335972,2.993921,...,0.0,-18.892036,-19.361521,80465.0,41.115579,1,0.452284,227.191787,36393.0,False
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,488.147784,430.622409,2.875093,...,0.0,-18.084681,-19.538399,85330.0,38.731811,1,0.356533,218.941201,30423.0,False
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,410.433901,1068.101440,1.147251,...,0.0,-19.307073,-19.974498,6688.0,20.848710,1,0.709480,24.215672,4745.0,False
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1224.321794,394.006757,3.063999,...,0.0,-18.892036,-19.635696,35500.0,34.298864,1,0.595127,111.924414,21127.0,False
4,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,420.393892,1091.808172,2.935847,...,0.0,-18.307073,-19.447251,64326.0,31.540664,0,0.255402,205.985812,16429.0,False


In [5]:
# Path to patient folders


# Default QC flags
sc_profiles_df["Metadata_cqc_organoid_flagged"] = False
sc_profiles_df["Metadata_cqc_nan_detected"] = (
    sc_profiles_df[
        ["Metadata_Object_ObjectID", "Nuclei_NoChannel_AreaSizeShape_Volume"]
    ]
    .isna()
    .any(axis=1)
)
sc_profiles_df["Metadata_cqc_missing_parent_organoid"] = (
    sc_profiles_df["Metadata_Object_ParentOrganoid"] == -1
)


organoid_flags_df = organoid_qc_profiles_df[
    ["Metadata_Object_ObjectID", "Metadata_Experiment_WellFOV"]
    + [col for col in organoid_qc_profiles_df.columns if col.startswith("Metadata_cqc")]
]

# Get flagged (object_id, image_set) pairs
flagged_pairs = set(
    organoid_flags_df.loc[
        organoid_flags_df.filter(like="cqc").any(axis=1),
        ["Metadata_Object_ObjectID", "Metadata_Experiment_WellFOV"],
    ].itertuples(index=False, name=None)
)

# Flag SC rows where both parent_organoid & image_set match a flagged organoid
sc_profiles_df["Metadata_cqc_organoid_flagged"] = sc_profiles_df.apply(
    lambda row: (
        (row["Metadata_Object_ParentOrganoid"], row["Metadata_Experiment_WellFOV"])
        in flagged_pairs
    ),
    axis=1,
)

print(sc_profiles_df.shape)
sc_profiles_df.head()

(190, 2651)


,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,Metadata_Location_Cell_CenterY,Metadata_Location_Cell_CenterZ,...,Nuclei_Mito_Texture_Variance-3-12-256,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_EquivalentDiameter,Nuclei_NoChannel_AreaSizeShape_EulerNumber,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei_NoChannel_AreaSizeShape_Volume,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1159.400905,258.335972,2.993921,...,-19.361521,80465.0,41.115579,1,0.452284,227.191787,36393.0,False,False,False
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,488.147784,430.622409,2.875093,...,-19.538399,85330.0,38.731811,1,0.356533,218.941201,30423.0,False,False,False
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,410.433901,1068.101440,1.147251,...,-19.974498,6688.0,20.848710,1,0.709480,24.215672,4745.0,False,False,False
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1224.321794,394.006757,3.063999,...,-19.635696,35500.0,34.298864,1,0.595127,111.924414,21127.0,False,False,False
4,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,420.393892,1091.808172,2.935847,...,-19.447251,64326.0,31.540664,0,0.255402,205.985812,16429.0,False,False,False


In [6]:
sc_profiles_df["Nuclei_NoChannel_AreaSizeShape_Volume"].describe()

count       190.000000
mean      55182.478947
std       38727.674307
min         630.000000
25%       16638.000000
50%       52804.500000
75%       85020.000000
max      149720.000000
Name: Nuclei_NoChannel_AreaSizeShape_Volume, dtype: float64

## Detect outlier single-cells using the non-flagged data

We will attempt to detect instances of poor quality segmentations using the nuclei compartment as the base. The conditions we are using are as follows:

1. Abnormally small or large nuclei using `Volume`
2. Abnormally high `mass displacement` in the nuclei for instances of mis-segmentation of background/no longer in-focus

In [7]:
# Set the metadata columns to be used in the QC process
metadata_columns = [x for x in sc_profiles_df.columns if "Metadata" in x]

In [8]:
# Only process the rows that are not flagged
filtered_plate_df = sc_profiles_df[
    ~(
        sc_profiles_df["Metadata_cqc_nan_detected"]
        | sc_profiles_df["Metadata_cqc_organoid_flagged"]
        | sc_profiles_df["Metadata_cqc_missing_parent_organoid"]
    )
]

# --- Find size based nuclei outliers ---
print("Finding small nuclei outliers...")
small_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_NoChannel_AreaSizeShape_Volume": -1,  # Detect very small nuclei
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_small_nuclei_outlier"] = False
sc_profiles_df.loc[small_nuclei_outliers.index, "Metadata_cqc_small_nuclei_outlier"] = (
    True
)

print("Finding large nuclei outliers...")
large_nuclei_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_NoChannel_AreaSizeShape_Volume": 2,  # Detect very large nuclei
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_large_nuclei_outlier"] = False
sc_profiles_df.loc[large_nuclei_outliers.index, "Metadata_cqc_large_nuclei_outlier"] = (
    True
)

# --- Find mass displacement based nuclei outliers ---
print("Finding high mass displacement outliers...")
high_mass_displacement_outliers = find_outliers(
    df=filtered_plate_df,
    metadata_columns=metadata_columns,
    feature_thresholds={
        "Nuclei_DNA_Intensity_MassDisplacement": 2,  # Detect high mass displacement
    },
)

# Ensure the column exists before assignment
sc_profiles_df["Metadata_cqc_mass_displacement_outlier"] = False
sc_profiles_df.loc[
    high_mass_displacement_outliers.index, "Metadata_cqc_mass_displacement_outlier"
] = True

# Print number of outliers (only in filtered rows)
small_count = filtered_plate_df.index.intersection(small_nuclei_outliers.index).shape[0]
large_count = filtered_plate_df.index.intersection(large_nuclei_outliers.index).shape[0]
high_mass_count = filtered_plate_df.index.intersection(
    high_mass_displacement_outliers.index
).shape[0]

print(f"Small nuclei outliers found: {small_count}")
print(f"Large nuclei outliers found: {large_count}")
print(f"High mass displacement outliers found: {high_mass_count}")

# Save updated plate_df with flag columns included
output_file_path = pathlib.Path(f"{output_dir}/sc_flagged_outliers.parquet").resolve()
sc_profiles_df.to_parquet(output_file_path, index=False)

Finding small nuclei outliers...
Number of outliers: 28 (23.93%)
Outliers Range:
Nuclei_NoChannel_AreaSizeShape_Volume Min: 1068.0
Nuclei_NoChannel_AreaSizeShape_Volume Max: 16429.0
Finding large nuclei outliers...
Number of outliers: 3 (2.56%)
Outliers Range:
Nuclei_NoChannel_AreaSizeShape_Volume Min: 139584.0
Nuclei_NoChannel_AreaSizeShape_Volume Max: 149720.0
Finding high mass displacement outliers...
Number of outliers: 3 (2.56%)
Outliers Range:
Nuclei_DNA_Intensity_MassDisplacement Min: 11.010803
Nuclei_DNA_Intensity_MassDisplacement Max: 15.393102
Small nuclei outliers found: 28
Large nuclei outliers found: 3
High mass displacement outliers found: 3


In [9]:
sc_profiles_df.head()

,Metadata_Biology_PatientTumor,Metadata_Experiment_Class,Metadata_Experiment_Target,Metadata_Experiment_TherapeuticCategories,Metadata_Experiment_Treatment,Metadata_Experiment_Well,Metadata_Experiment_WellFOV,Metadata_Location_Cell_CenterX,Metadata_Location_Cell_CenterY,Metadata_Location_Cell_CenterZ,...,Nuclei_NoChannel_AreaSizeShape_EulerNumber,Nuclei_NoChannel_AreaSizeShape_Extent,Nuclei_NoChannel_AreaSizeShape_SurfaceArea,Nuclei_NoChannel_AreaSizeShape_Volume,Metadata_cqc_nan_detected,Metadata_cqc_organoid_flagged,Metadata_cqc_missing_parent_organoid,Metadata_cqc_small_nuclei_outlier,Metadata_cqc_large_nuclei_outlier,Metadata_cqc_mass_displacement_outlier
0,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1159.400905,258.335972,2.993921,...,1,0.452284,227.191787,36393.0,False,False,False,False,False,False
1,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,488.147784,430.622409,2.875093,...,1,0.356533,218.941201,30423.0,False,False,False,False,False,False
2,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,410.433901,1068.101440,1.147251,...,1,0.709480,24.215672,4745.0,False,False,False,True,False,False
3,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,1224.321794,394.006757,3.063999,...,1,0.595127,111.924414,21127.0,False,False,False,False,False,False
4,NF0014_T1,Small Molecule,Apoptosis induction,Experimental,Staurosporine 10 nM,C2,C2-1,420.393892,1091.808172,2.935847,...,0,0.255402,205.985812,16429.0,False,False,False,True,False,False
